# Drone environment

Let's setup the installation

In [1]:

######
# On colab
######

#!pip install pybullet
#!pip install git+https://github.com/utiasDSL/gym-pybullet-drones.git
#!pip install torchrl tensordict gymnasium==0.29.1

######
# On your own computer
######

# create a new environment
#conda create -n drone_env python=3.10
#conda activate drone_env

# install pybullet 
#pip install pybullet
#pip install pybullet --no-cache-dir #if on Mac Silicon

#install drone library
#pip install git+https://github.com/utiasDSL/gym-pybullet-drones.git

# install pytorch 
# for CPU only version:
#pip install torch torchvision

# install torchrl and tensordict
#pip install torchrl tensordict

# install gymnasium
#pip install gymnasium==0.29.1

# verify everything works
#python -c "from gym_pybullet_drones.envs.HoverAviary import HoverAviary; print('drone env ok')"
#python -c "import torchrl; print('torchrl ok')"

Let's create the drone environment

In [1]:
# pip install git+https://github.com/utiasDSL/gym-pybullet-drones.git

import torch
from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ObservationType, ActionType
from torchrl.envs.libs.gym import GymWrapper
from torchrl.envs import TransformedEnv, Compose, ObservationNorm, DoubleToFloat, StepCounter

# instantiate directly as a class, not via gym.make()
raw_env = HoverAviary(obs=ObservationType('kin'), act=ActionType('one_d_rpm'))

# wrap in TorchRL
base_env = GymWrapper(raw_env)

# estimate normalisation stats from raw environment
raw_rollout = GymWrapper(
    HoverAviary(obs=ObservationType('kin'), act=ActionType('one_d_rpm'))
).rollout(1000)
loc   = raw_rollout["observation"].mean(0)
scale = raw_rollout["observation"].std(0)

# build transformed env
env = TransformedEnv(
    base_env,
    Compose(
        ObservationNorm(in_keys=["observation"], loc=loc, scale=scale, standard_normal=True),
        DoubleToFloat(),
        StepCounter(),
    )
)

# verify normalisation
check = env.rollout(200)
print(f"Mean: {check['observation'].mean(0)}")
print(f"Std:  {check['observation'].std(0)}")
print(f"Obs shape: {check['observation'].shape}")
print(f"Action shape: {check['action'].shape}")

/Users/tyler/miniforge3/envs/drone_env/lib/python3.10/site-packages/gym_pybullet_drones/envs/BaseAviary.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
pybullet build time: Apr 23 2026 11:15:38
/Users/tyler/miniforge3/envs/drone_env/lib/python3.10/site-packages/torchrl/data/replay_buffers/samplers.py:37: UserWarning: Failed to import torchrl C++ binaries. Some modules (eg, prioritized replay buffers) may not work with your installation. This is likely due to a discrepancy between your package version and the PyTorch version. Make sure both are compatible. Usually, torchrl majors follow the pytorch majors within a few days around the release. For instance, TorchRL 0.5 requires PyTorch 2.4.0, and TorchRL 0.6 requires PyTorch 2.5.0.
  warnings.warn(EXTENSION_WARNING)


[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 3.160000e-10, km 7.940000e-12,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 3.160000e-10, km 7.940000e-12,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
Mean: tensor([[    nan,     nan, -0.9925,     nan,     nan,     nan,     nan,     nan,
         -2.2415,     nan,     nan,     nan, -0.0984, -0.0968, -0.0967, -0.

Let's visualize the drone environment... (crashes when finished ... )

In [ ]:
# instantiate with GUI for visualization
viz_env = HoverAviary(
    obs=ObservationType('kin'),
    act=ActionType('one_d_rpm'),
    gui=True
)

obs, info = viz_env.reset()

# run a random rollout and render each step
for i in range(300):
    action = viz_env.action_space.sample()
    obs, reward, terminated, truncated, info = viz_env.step(action)
    viz_env.render()
    if terminated or truncated:
        obs, info = viz_env.reset()

viz_env.close()

[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 3.160000e-10, km 7.940000e-12,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
Version = 4.1 Metal - 76.3
Vendor = Apple
Renderer = Apple M1 Pro
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
viewMatrix (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
projectionMatrix (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)

[INFO] BaseAviary.render() ——— it 0008 ——— wall-clock time 0.7s, simulation time 0.0s@240Hz (0.05x)
[INFO] BaseAviary.render() ——— drone 0 ——— x +00.00, y +00.00, z +00.11 ——— velocit